# Multimodal RAG — CLIP + BLIP + Corrective RAG

نوتبوك رفيع: كل المنطق داخل حزمة `multimodal_rag/` على GitHub، والنوتبوك ده بس بيجيبها ويشغّلها.

**قبل ما تشغّل:** غيّر `REPO_URL` تحت بالرابط بتاع الـ repo بتاعك (شوف README.md في الحزمة لخطوات الرفع على GitHub).

In [ ]:
REPO_URL = "https://github.com/<username>/<repo-name>.git"  # <-- غيّر ده
REPO_DIR = "multimodal_rag_repo"


## 1) Clone + install

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull


In [ ]:
%cd {REPO_DIR}
!pip install -q -r requirements.txt


In [ ]:
import sys
sys.path.insert(0, os.getcwd())


## 2) API key + تحميل الحزمة

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

import multimodal_rag as mrag
mrag.init_client(GOOGLE_API_KEY)


أول استدعاء لـ `ask()` هيبني الـ index (CLIP embeddings لكل الـ 50 نص + 12 صورة) — بياخد لحظات أول مرة بس، وبعدين بيتخزن في الذاكرة.

## 3) مثال: Text query

In [ ]:
result = mrag.ask("What do giraffes eat and how does their long neck help them?")

print("Answer:\n", result["answer"])
print("\nAttempts:", result["attempts"])
if "warning" in result:
    print("WARNING:", result["warning"])

print("\n--- Retrieved items ---")
for item, score in result["results"]:
    print(f"[{item.type}] score={score:.3f} -> {item.source.get('animal')}: "
          f"{item.content if item.type=='text' else item.content}")


## 4) مثال: Image query

ارفع صورة (مثلاً من `data/images/`) واسأل عليها.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image_path = "data/images/elephant1.jpg"  # جرّب أي صورة من data/images

plt.imshow(Image.open(image_path))
plt.axis("off")
plt.show()

result = mrag.ask(
    "What animal is this and what is interesting about it?",
    image_path=image_path,
)

print("Answer:\n", result["answer"])
print("\nAttempts:", result["attempts"])


## 5) مثال: PDF query

ارفع أي PDF على Colab (زر الملفات على الشمال) وغيّر المسار تحت.

In [ ]:
pdf_path = "/content/your_file.pdf"  # غيّر ده لمسار الـ PDF اللي رفعته

result = mrag.ask(
    "Summarize the main idea of the uploaded document.",
    pdf_path=pdf_path,
)

print("Answer:\n", result["answer"])


## 6) قراءة تفاصيل الـ Corrective RAG

كل محاولة (attempt) وليه اترفضت أو اتقبلت متسجلة في `trace`.

In [ ]:
for step in result["trace"]:
    print(f"Attempt {step['attempt']} | correct={step['graded_correct']}")
    print("query:", step["search_query"])
    print("answer:", step["answer"][:200], "...")
    print("---")
